In [1]:
import numpy as np
import networkx as nx
import pickle
import os
import json

from utils import *

%matplotlib inline
%load_ext autoreload
%autoreload 2

setup_environment()
set_paper_style()

In [2]:
import json
import pandas as pd

# --- 設定 ---
INPUT_JSON_PATH = "knowledge_graph_with_wiki.json"
OUTPUT_NODES_CSV = "graphrag_project/input/nodes.csv"
OUTPUT_EDGES_CSV = "graphrag_project/input/edges.csv"

# --- 知識グラフの読み込み ---
with open(INPUT_JSON_PATH, "r", encoding="utf-8") as f:
    graph_data = json.load(f)

# --- ノードデータの作成 ---
node_records = []
for node in graph_data["nodes"]:
    if node.get("type") == "市区町村" and node.get("summary_text"):
        # if node.get("type") == "市区町村":
        node_records.append(
            {
                "id": node["id"],
                "label": node.get("label"),
                "summary_text": node.get("summary_text"),
            }
        )
df_nodes = pd.DataFrame(node_records)
df_nodes.to_csv(OUTPUT_NODES_CSV, index=False, encoding="utf-8")
print(f"✅ {len(df_nodes)}件のノードデータを '{OUTPUT_NODES_CSV}' に保存しました。")

# --- エッジデータの作成 ---
edge_records = []
for edge in graph_data["links"]:
    if edge.get("label") == "IS_IN":
        edge_records.append(
            {
                "source": edge["source"],
                "target": edge["target"],
                "description": f"{edge['source']}は{edge['target']}に属しています。",
            }
        )
df_edges = pd.DataFrame(edge_records)
df_edges.to_csv(OUTPUT_EDGES_CSV, index=False, encoding="utf-8")
print(f"✅ {len(df_edges)}件のエッジデータを '{OUTPUT_EDGES_CSV}' に保存しました。")

✅ 1741件のノードデータを 'graphrag_project/input/nodes.csv' に保存しました。
✅ 1741件のエッジデータを 'graphrag_project/input/edges.csv' に保存しました。


In [ ]:
import subprocess
import os

# https://microsoft.github.io/graphrag/config/models/
# 約9時間ぐらいかかってて草(o4-mini)
# ==============================================================================
# 設定
# ==============================================================================
# GraphRagプロジェクトのルートフォルダ
PROJECT_ROOT = "./graphrag_project"

# ユーザーからの質問
USER_QUESTION = "温泉地として有名で、自然が豊かで静かに暮らせそうな街はどこですか？"
# ==============================================================================


def run_command(command):
    """
    ターミナルコマンドを実行し、出力をリアルタイムで表示する関数
    """
    print(f"▶️ 実行中: {' '.join(command)}\n")
    # `subprocess.run` を使用してコマンドを実行
    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        encoding="utf-8",
        # .envファイルから環境変数を読み込むために、PROJECT_ROOTをカレントディレクトリとして指定
        cwd=os.path.abspath(PROJECT_ROOT),
    )

    # 標準出力と標準エラー出力を表示
    if result.stdout:
        print("--- 出力 ---")
        print(result.stdout)
    if result.stderr:
        print("--- エラー出力 ---")
        print(result.stderr)

    if result.returncode == 0:
        print("\n✅ コマンドは正常に完了しました。")
    else:
        print(
            f"\n❌ コマンドの実行中にエラーが発生しました (終了コード: {result.returncode})。"
        )

    return result


# --- 1. インデックスの作成 ---
print("🔄 インデックス作成を開始します...（データ量に応じて時間がかかります）")
# 正しいコマンド: `graphrag index`
index_command = ["graphrag", "index", "--root", "."]
run_command(index_command)

🔄 インデックス作成を開始します...（データ量に応じて時間がかかります）
▶️ 実行中: graphrag index --root .

--- 出力 ---
2025-08-14 02:05:39.0513 - INFO - graphrag.cli.index - Logging enabled at /Users/tanno/tanno/mygit/ssdse2025/graphrag_project/logs/logs.txt
2025-08-14 02:05:42.0331 - INFO - graphrag.index.validate_config - LLM Config Params Validated
2025-08-14 02:05:43.0397 - INFO - graphrag.index.validate_config - Embedding LLM Config Params Validated
2025-08-14 02:05:43.0397 - INFO - graphrag.cli.index - Starting pipeline run. False
2025-08-14 02:05:43.0398 - INFO - graphrag.cli.index - Using default configuration: {
    "root_dir": "/Users/tanno/tanno/mygit/ssdse2025/graphrag_project",
    "models": {
        "default_chat_model": {
            "api_key": "==== REDACTED ====",
            "auth_type": "api_key",
            "type": "azure_openai_chat",
            "model": "gpt-4o-mini",
            "encoding_model": "o200k_base",
            "api_base": "https://rtan-m8ltk59u-eastus2.openai.azure.com/",
            

CompletedProcess(args=['graphrag', 'index', '--root', '.'], returncode=0, stdout='2025-08-14 02:05:39.0513 - INFO - graphrag.cli.index - Logging enabled at /Users/tanno/tanno/mygit/ssdse2025/graphrag_project/logs/logs.txt\n2025-08-14 02:05:42.0331 - INFO - graphrag.index.validate_config - LLM Config Params Validated\n2025-08-14 02:05:43.0397 - INFO - graphrag.index.validate_config - Embedding LLM Config Params Validated\n2025-08-14 02:05:43.0397 - INFO - graphrag.cli.index - Starting pipeline run. False\n2025-08-14 02:05:43.0398 - INFO - graphrag.cli.index - Using default configuration: {\n    "root_dir": "/Users/tanno/tanno/mygit/ssdse2025/graphrag_project",\n    "models": {\n        "default_chat_model": {\n            "api_key": "==== REDACTED ====",\n            "auth_type": "api_key",\n            "type": "azure_openai_chat",\n            "model": "gpt-4o-mini",\n            "encoding_model": "o200k_base",\n            "api_base": "https://rtan-m8ltk59u-eastus2.openai.azure.com/",

In [33]:
import subprocess
import os

# ==============================================================================
# 設定
# ==============================================================================
PROJECT_ROOT = "./graphrag_project"
USER_QUESTION = "現在、福岡市に住んでいて福岡市の良いところを維持しながら２拠点生活をしたと考えている。また、飛行機を利用したいと思っているため、九州以外での２拠点都市を考えている。特徴としては、温泉地として有名で、自然が豊かで静かに暮らせそうな街はどこですか？"

# ==============================================================================
print(f"\n【質問】: {USER_QUESTION}")
print("🔄 GraphRagで検索を実行しています...")

# '--query' オプションを質問文の前に追加
query_command = [
    "graphrag",
    "query",
    "--root",
    ".",
    "--method",
    "global",
    "--query",
    USER_QUESTION,
]
query_result = run_command(query_command)


【質問】: 現在、福岡市に住んでいて福岡市の良いところを維持しながら２拠点生活をしたと考えている。また、飛行機を利用したいと思っているため、九州以外での２拠点都市を考えている。特徴としては、温泉地として有名で、自然が豊かで静かに暮らせそうな街はどこですか？
🔄 GraphRagで検索を実行しています...
▶️ 実行中: graphrag query --root . --method global --query 現在、福岡市に住んでいて福岡市の良いところを維持しながら２拠点生活をしたと考えている。また、飛行機を利用したいと思っているため、九州以外での２拠点都市を考えている。特徴としては、温泉地として有名で、自然が豊かで静かに暮らせそうな街はどこですか？

--- 出力 ---
2025-08-14 12:35:41.0934 - INFO - graphrag.storage.file_pipeline_storage - Creating file storage at /Users/tanno/tanno/mygit/ssdse2025/graphrag_project/output
2025-08-14 12:35:41.0935 - INFO - graphrag.utils.storage - reading table from storage: entities.parquet
2025-08-14 12:35:41.0960 - INFO - graphrag.utils.storage - reading table from storage: communities.parquet
2025-08-14 12:35:41.0969 - INFO - graphrag.utils.storage - reading table from storage: community_reports.parquet
2025-08-14 12:36:10.0982 - INFO - graphrag.cli.query - Global Search Response:
## 九州以外での二拠点生活に適した温泉地

福岡市に住みながら、九州以外での二拠点生活を考える際、温泉地として有名で自然が豊かで静かな環境を提供する都市はいくつかあります。以下に、特

In [30]:
# --- 3. 結果の表示 ---
# 検索結果は`query_result.stdout`に格納されている
print("\n\n===================================")
print("      ✨ GraphRagによる回答 ✨")
print("===================================")
# 回答はJSON形式で返ってくることが多いので、そのまま表示
print(query_result.stdout)



      ✨ GraphRagによる回答 ✨
2025-08-14 12:28:40.0089 - INFO - graphrag.storage.file_pipeline_storage - Creating file storage at /Users/tanno/tanno/mygit/ssdse2025/graphrag_project/output
2025-08-14 12:28:40.0090 - INFO - graphrag.utils.storage - reading table from storage: entities.parquet
2025-08-14 12:28:40.0119 - INFO - graphrag.utils.storage - reading table from storage: communities.parquet
2025-08-14 12:28:40.0128 - INFO - graphrag.utils.storage - reading table from storage: community_reports.parquet
2025-08-14 12:29:02.0968 - INFO - graphrag.cli.query - Global Search Response:
## 温泉地として有名な静かな街

日本には多くの温泉地があり、自然が豊かで静かに暮らせる街がいくつか存在します。以下に、特に注目すべき地域をいくつか紹介します。

### 1. 指宿市 (Ibusuki City)
指宿市は、独特の砂風呂で知られ、自然の美しさに囲まれた理想的な生活環境を提供しています。この地域の温泉はリラクゼーションと健康を求める多くの訪問者を引き寄せており、静かな生活を求める人々にとって魅力的な場所です [Data: Reports (926, 922)]。

### 2. 湯河原町 (Yugawara Town)
神奈川県に位置する湯河原町は、湯河原温泉で有名です。自然が豊かで、静かな環境での生活が可能です。温泉はリラクゼーションと癒しを提供し、地域経済にも貢献しています [Data: Reports (299)]。

### 3. 新温泉町 (Shinonsen Town)
兵庫県の北部